# Étape 2 — Découverte : embeddings et recherche par similarité

Objectif : comprendre **à la main** les deux briques du RAG avant de coder le pipeline.

1. Transformer des phrases en vecteurs (*embeddings*) avec Ollama (`nomic-embed-text`).
2. Mesurer la proximité de sens entre phrases avec la **similarité cosinus**, calculée nous-mêmes.
3. Stocker des vecteurs dans **Qdrant** et lancer une recherche.

**Prérequis** : `docker compose up -d` lancé (Ollama et Qdrant tournent), puis sur le PC :

```
pip install jupyter httpx numpy qdrant-client
```

Le notebook tourne sur le PC (hors Docker) : on joint donc les services par `localhost`.

In [ ]:
import httpx
import numpy as np

OLLAMA_URL = "http://localhost:11434"
QDRANT_URL = "http://localhost:6333"
EMBED_MODEL = "nomic-embed-text"

print(httpx.get(OLLAMA_URL).text)  # doit afficher « Ollama is running »

## 1. Calculer des embeddings avec Ollama

Un **embedding** est un vecteur de nombres qui représente le *sens* d'un texte. Le modèle `nomic-embed-text` renvoie des vecteurs de **768 dimensions**.

On appelle l'endpoint `POST /api/embed` d'Ollama. Le préfixe `search_document:` est recommandé par les auteurs du modèle pour les textes indexés (et `search_query:` pour les questions).

In [ ]:
def embed(texts: list[str], prefix: str = "search_document: ") -> np.ndarray:
    response = httpx.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": EMBED_MODEL, "input": [prefix + t for t in texts]},
        timeout=120,
    )
    response.raise_for_status()
    return np.array(response.json()["embeddings"])


phrases = [
    "La moyenne est la somme des valeurs divisée par leur nombre.",
    "On calcule la moyenne en additionnant les valeurs puis en divisant par l'effectif.",
    "Le chat dort sur le canapé du salon.",
]
vecteurs = embed(phrases)
print("Forme :", vecteurs.shape)  # (3, 768)
print("Début du 1er vecteur :", vecteurs[0][:8].round(3))

## 2. Similarité cosinus, calculée à la main

La similarité cosinus mesure l'angle entre deux vecteurs :

$$\cos(u, v) = \frac{u \cdot v}{\lVert u \rVert \times \lVert v \rVert}$$

- proche de **1** : même direction, donc sens proche ;
- plus faible : sens éloigné.

On s'attend à ce que les phrases 1 et 2 (même idée, mots différents) soient plus proches que les phrases 1 et 3.

In [ ]:
def cosinus(u: np.ndarray, v: np.ndarray) -> float:
    produit_scalaire = sum(a * b for a, b in zip(u, v))  # u · v
    norme_u = sum(a * a for a in u) ** 0.5  # ||u||
    norme_v = sum(b * b for b in v) ** 0.5  # ||v||
    return produit_scalaire / (norme_u * norme_v)


for i in range(len(phrases)):
    for j in range(i + 1, len(phrases)):
        print(f"phrase {i + 1} / phrase {j + 1} : {cosinus(vecteurs[i], vecteurs[j]):.3f}")

In [ ]:
# Même calcul avec numpy (matrice de similarité complète) : vérification
normes = np.linalg.norm(vecteurs, axis=1, keepdims=True)
matrice = (vecteurs / normes) @ (vecteurs / normes).T
print(matrice.round(3))

**Observation** *(à compléter avec tes mots)* : les phrases 1 et 2 ont une similarité plus élevée que les phrases 1 et 3, alors qu'elles n'emploient presque pas les mêmes mots. L'embedding capture donc le **sens**, pas seulement les mots communs : c'est ce qui permet de retrouver un passage de cours même si la question est formulée autrement.

## 3. Recherche par similarité avec Qdrant

Qdrant est une **base vectorielle** : elle stocke des vecteurs (+ des métadonnées, le *payload*) et retrouve très vite les plus proches d'un vecteur donné.

On crée une collection de test, on y insère quelques phrases de cours, puis on cherche avec une question.

In [ ]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url=QDRANT_URL)
COLLECTION = "decouverte"

passages = [
    ("La médiane partage la série ordonnée en deux groupes de même effectif.", "Statistiques"),
    ("L'écart-type est la racine carrée de la variance.", "Statistiques"),
    ("Le coefficient de corrélation de Pearson est compris entre -1 et 1.", "Statistiques"),
    ("Une liste Python est modifiable, un tuple ne l'est pas.", "Python"),
    ("La méthode groupby de pandas regroupe les lignes avant une agrégation.", "Python"),
]
vecteurs_passages = embed([texte for texte, _ in passages])

if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)
client.create_collection(
    COLLECTION,
    vectors_config=models.VectorParams(
        size=vecteurs_passages.shape[1], distance=models.Distance.COSINE
    ),
)
client.upsert(
    COLLECTION,
    points=[
        models.PointStruct(id=i, vector=v.tolist(), payload={"texte": t, "matiere": m})
        for i, ((t, m), v) in enumerate(zip(passages, vecteurs_passages))
    ],
)
print(client.count(COLLECTION))

In [ ]:
def rechercher(question: str, k: int = 3, matiere: str | None = None):
    vecteur_question = embed([question], prefix="search_query: ")[0]
    filtre = None
    if matiere:
        filtre = models.Filter(
            must=[models.FieldCondition(key="matiere", match=models.MatchValue(value=matiere))]
        )
    resultats = client.query_points(
        COLLECTION, query=vecteur_question.tolist(), limit=k, query_filter=filtre
    )
    for point in resultats.points:
        print(f"{point.score:.3f}  [{point.payload['matiere']}]  {point.payload['texte']}")


rechercher("Comment mesurer la dispersion des données ?")

In [ ]:
# Avec un filtre sur la matière (métadonnée stockée dans le payload)
rechercher("Quelle structure de données ne peut pas être modifiée ?", matiere="Python")

## 4. Ce que j'ai compris (à rédiger avec mes mots)

- **Embedding** : …
- **Similarité cosinus** : …
- **Recherche par similarité** : …
- **Lien avec le RAG** : la question est transformée en vecteur avec le *même* modèle que les documents ; Qdrant renvoie les *k* passages les plus proches ; ces passages sont donnés au LLM comme contexte pour qu'il réponde sans inventer.

Tu peux supprimer la collection de test depuis le tableau de bord : http://localhost:6333/dashboard